# LeetCode 3406 — Find the Lexicographically Largest String From the Box II bằng Python

- Link: https://leetcode.com/problems/find-the-lexicographically-largest-string-from-the-box-ii/
- Độ khó: **Hard** · **Premium**
- Chủ đề: **Hậu tố lớn nhất theo thứ tự từ điển** (thuật toán hai con trỏ `i, j, k` — LC 1163), **cắt độ dài** theo số phần cần chia

> **Ghi chú Premium:** bài này là Premium nên LeetCode không trả về đề qua API công khai. Đề, ví dụ, ràng buộc và chữ ký `answerString(self, word, numFriends)` lấy theo trang doocs (https://leetcode.doocs.org/en/lc/3406/) và trùng chữ ký bản I (LC 3403). Vui lòng **đối chiếu lại** nếu có tài khoản Premium.

Skeleton:

```python
class Solution:
    def answerString(self, word: str, numFriends: int) -> str:
```

## 1. Mục tiêu học

1. Mô tả chính xác tập các xâu có thể xuất hiện trong hộp: mọi xâu con `word[i:j]` mà phần còn lại đủ chia thành `numFriends − 1` mảnh khác rỗng.
2. Chứng minh ứng viên tốt nhất bắt đầu ở `i` là xâu **dài nhất** có thể: `word[i : i + min(n − i, n − k + 1)]`.
3. Chứng minh đáp án là **tiền tố độ dài `n − k + 1` của hậu tố lớn nhất** (với `k ≥ 2`), và `k = 1` thì đáp án là cả `word`.
4. Cài thuật toán **hậu tố lớn nhất** `O(n)` bằng hai con trỏ (bài LC 1163).
5. Cách thay thế: so sánh hậu tố bằng **băm + nhị phân LCP** `O(n log n)`.

## 2. Đề bài & ví dụ

Cho `word` và `numFriends = k`. Mỗi vòng, `word` được chia thành đúng `k` xâu **khác rỗng** (mỗi cách chia chỉ dùng một lần); mọi mảnh được bỏ vào hộp. Sau khi đã dùng **mọi** cách chia, trả về xâu **lớn nhất theo thứ tự từ điển** trong hộp.

**Ví dụ 1:** `word = "dbca", k = 2` → `"dbc"` (các cách chia: `d|bca`, `db|ca`, `dbc|a`).

**Ví dụ 2:** `word = "gggg", k = 4` → `"g"`.

**Ràng buộc (theo doocs):** `1 ≤ n ≤ 2·10^5`, `1 ≤ k ≤ n`, chữ thường.

In [1]:
examples = [("dbca", 2, "dbc"), ("gggg", 4, "g")]
for w, k, exp in examples:
    print(repr(w), k, '-> kỳ vọng', repr(exp))

'dbca' 2 -> kỳ vọng 'dbc'
'gggg' 4 -> kỳ vọng 'g'


## 3. Kiến thức nền

### 3.1. Xâu nào có thể là một mảnh?

`word[i:j]` (khác rỗng) là một mảnh trong **một** cách chia nào đó ⇔ phần trái (`i` ký tự) và phần phải (`n − j` ký tự) có thể chia thành tổng cộng `k − 1` mảnh khác rỗng. Phần rỗng dùng 0 mảnh, phần khác rỗng cần **ít nhất 1** mảnh và **nhiều nhất** bằng số ký tự của nó ⇒ điều kiện:

`[i > 0] + [j < n]  ≤  k − 1  ≤  i + (n − j)`.

Ví dụ `"zab"`, `k = 2`: chữ `"a"` ở giữa **không** là mảnh được (hai phía đều khác rỗng cần 2 mảnh > 1).

In [2]:
from itertools import combinations

def all_pieces(word, k):
    n = len(word)
    out = set()
    for cuts in combinations(range(1, n), k - 1):
        b = (0,) + cuts + (n,)
        for a, c in zip(b, b[1:]):
            out.add(word[a:c])
    return out

def pieces_by_rule(word, k):
    n = len(word)
    return {word[i:j] for i in range(n) for j in range(i + 1, n + 1)
            if (i > 0) + (j < n) <= k - 1 <= i + (n - j)}

ok = all(all_pieces(w, k) == pieces_by_rule(w, k) for w in ["dbca", "zab", "abcab", "aaaa"] for k in range(1, len(w) + 1))
print('quy tắc khớp liệt kê:', ok, '|', sorted(all_pieces("dbca", 2)), '| "zab", k=2:', sorted(all_pieces("zab", 2)))

quy tắc khớp liệt kê: True | ['a', 'bca', 'ca', 'd', 'db', 'dbc'] | "zab", k=2: ['ab', 'b', 'z', 'za']


### 3.2. So sánh từ điển và tiền tố

Nếu `a` là tiền tố thật của `b` thì `a < b`. Vậy với điểm bắt đầu `i` cố định, xâu **dài nhất** được phép là lớn nhất. Độ dài tối đa: `min(n − i, n − k + 1)` (khi lấy hết phần bên phải, phần trái `i` ký tự phải đủ cho `k − 1` mảnh; nếu `i < k − 1` thì cắt bớt bên phải: `j = n − (k − 1 − i)`, độ dài `n − k + 1`).

In [3]:
print("ab" < "abc", "abz" > "abc", "b" > "abz")

True True True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `"dbca"`, `k = 2`, `L = n − k + 1 = 3`:

- `i = 0`: `"dbc"`; `i = 1`: `"bca"`; `i = 2`: `"ca"`; `i = 3`: `"a"` ⇒ lớn nhất `"dbc"`.

Hậu tố lớn nhất của `"dbca"` là `"dbca"`; cắt còn 3 ký tự: `"dbc"` ✓.

Ví dụ 2 `"gggg"`, `k = 4`, `L = 1`: mọi ứng viên là `"g"`.

In [4]:
w, k = "dbca", 2
L = len(w) - k + 1
print([w[i:i + min(len(w) - i, L)] for i in range(len(w))])

['dbc', 'bca', 'ca', 'a']


## 5. Cách 1 — Brute Force: liệt kê mọi cách chia

In [5]:
def answer_brute(word, k):
    return max(all_pieces(word, k))

for w, k, exp in examples:
    print(repr(answer_brute(w, k)), '| kỳ vọng', repr(exp))

'dbc' | kỳ vọng 'dbc'
'g' | kỳ vọng 'g'


**Độ phức tạp:** `C(n − 1, k − 1)` cách chia × `O(n)` — chỉ cho `n ≤ 12`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bổ đề.** Với `k ≥ 2`, `L = n − k + 1`: đáp án = `S[:L]` với `S` là hậu tố lớn nhất của `word`.

*Chứng minh:* ứng viên tại `i` là `t_i = word[i : i + min(n − i, L)]` = tiền tố độ dài `≤ L` của hậu tố `word[i:]`. Với hậu tố lớn nhất `S = word[i*:]` và bất kỳ `i`: nếu `word[i:]` khác `S` tại vị trí `p < L` thì `t_i < t_{i*}`; nếu khác tại `p ≥ L` thì hai tiền tố độ dài `L` bằng nhau; nếu `word[i:]` là tiền tố của `S` thì `t_i` là tiền tố của `t_{i*}`. Mọi trường hợp `t_i ≤ t_{i*}`.

**Hậu tố lớn nhất trong `O(n)`** (LC 1163): giữ ứng viên `i`, đối thủ `j`, độ dài trùng `k`. So `s[i+k]` với `s[j+k]`: bằng ⇒ `k += 1`; `s[i+k] < s[j+k]` ⇒ mọi điểm bắt đầu `i..i+k` đều thua ⇒ `i = i + k + 1` (nếu `i ≥ j` thì `j = i + 1`); ngược lại `j = j + k + 1`. Mỗi bước làm `i + j + k` tăng ⇒ `O(n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Thử mọi điểm bắt đầu, lấy max (`O(n · L)`)

In [6]:
def answer_candidates(word, k):
    if k == 1:
        return word
    n = len(word)
    L = n - k + 1
    return max(word[i:i + min(n - i, L)] for i in range(n))

for w, k, exp in examples:
    print(repr(answer_candidates(w, k)), '| kỳ vọng', repr(exp))

'dbc' | kỳ vọng 'dbc'
'g' | kỳ vọng 'g'


### 7.2. Cách 3 — Hậu tố lớn nhất bằng hai con trỏ (lời giải chính)

In [7]:
def last_substring_start(s):
    n = len(s)
    i, j, k = 0, 1, 0
    while j + k < n:
        a, b = s[i + k], s[j + k]
        if a == b:
            k += 1
        elif a < b:
            i += k + 1
            k = 0
            if i >= j:
                j = i + 1
        else:
            j += k + 1
            k = 0
    return i

def answer_two_pointers(word, k):
    if k == 1:
        return word
    i = last_substring_start(word)
    return word[i:i + len(word) - k + 1]

print('hậu tố lớn nhất của "leetcode":', "leetcode"[last_substring_start("leetcode"):], '| "abab":', "abab"[last_substring_start("abab"):])
for w, k, exp in examples:
    print(repr(answer_two_pointers(w, k)), '| kỳ vọng', repr(exp))

hậu tố lớn nhất của "leetcode": tcode | "abab": bab
'dbc' | kỳ vọng 'dbc'
'g' | kỳ vọng 'g'


### 7.3. Cách 4 — So sánh hậu tố bằng băm đa thức + nhị phân LCP (`O(n log n)`)

Tính băm tiền tố; so sánh hai hậu tố bằng cách nhị phân độ dài tiền tố chung dài nhất (LCP) rồi so ký tự kế tiếp. Lấy max qua `n` hậu tố bằng `n − 1` phép so sánh.

In [8]:
def answer_hash(word, k):
    if k == 1:
        return word
    n = len(word)
    MOD, B = (1 << 61) - 1, 911382323
    h = [0] * (n + 1); pw = [1] * (n + 1)
    for i, ch in enumerate(word):
        h[i + 1] = (h[i] * B + ord(ch)) % MOD
        pw[i + 1] = pw[i] * B % MOD
    def get(i, length):
        return (h[i + length] - h[i] * pw[length]) % MOD
    def greater(i, j):                     # hậu tố i > hậu tố j ?
        lo, hi = 0, min(n - i, n - j)
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if get(i, mid) == get(j, mid): lo = mid
            else: hi = mid - 1
        if lo == min(n - i, n - j):
            return n - i > n - j
        return word[i + lo] > word[j + lo]
    best = 0
    for i in range(1, n):
        if greater(i, best):
            best = i
    return word[best:best + n - k + 1]

for w, k, exp in examples:
    print(repr(answer_hash(w, k)), '| kỳ vọng', repr(exp))

'dbc' | kỳ vọng 'dbc'
'g' | kỳ vọng 'g'


## 8. TRACE từng bước

Trace hai con trỏ trên `"cacacb"`.

In [9]:
def trace(s):
    n = len(s)
    i, j, k = 0, 1, 0
    while j + k < n:
        a, b = s[i + k], s[j + k]
        if a == b:
            act = 'bằng -> k += 1'; k += 1
        elif a < b:
            act = f's[i+k]={a!r} < s[j+k]={b!r} -> i = {i + k + 1}'; i += k + 1; k = 0
            if i >= j:
                j = i + 1; act += f', j = {j}'
        else:
            act = f's[i+k]={a!r} > s[j+k]={b!r} -> j = {j + k + 1}'; j += k + 1; k = 0
        print(f"i={i}, j={j}, k={k} | {act}")
    print('hậu tố lớn nhất:', s[i:], '| kiểm tra max:', max(s[t:] for t in range(n)))

trace("cacacb")

i=0, j=2, k=0 | s[i+k]='c' > s[j+k]='a' -> j = 2
i=0, j=2, k=1 | bằng -> k += 1
i=0, j=2, k=2 | bằng -> k += 1
i=0, j=2, k=3 | bằng -> k += 1
i=4, j=5, k=0 | s[i+k]='a' < s[j+k]='b' -> i = 4, j = 5
i=4, j=6, k=0 | s[i+k]='c' > s[j+k]='b' -> j = 6
hậu tố lớn nhất: cb | kiểm tra max: cb


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def answerString(self, word: str, numFriends: int) -> str:
        if numFriends == 1:
            return word
        n = len(word)
        i, j, k = 0, 1, 0
        while j + k < n:
            a, b = word[i + k], word[j + k]
            if a == b:
                k += 1
            elif a < b:
                i += k + 1
                k = 0
                if i >= j:
                    j = i + 1
            else:
                j += k + 1
                k = 0
        return word[i:i + n - numFriends + 1]

for w, k, exp in examples:
    got = Solution().answerString(w, k)
    print(repr(got), 'OK' if got == exp else 'SAI')

'dbc' OK
'g' OK


## 10. Vì sao thuật toán đúng

1. **Tập mảnh (3.1)** và **ứng viên tốt nhất theo điểm bắt đầu (3.2)**.
2. **Bổ đề (mục 6)** ⇒ đáp án là tiền tố độ dài `L` của hậu tố lớn nhất (`k ≥ 2`). `k = 1` ⇒ chỉ có một mảnh là cả xâu.
3. **Thuật toán hai con trỏ:** bất biến — hậu tố lớn nhất bắt đầu tại `i` hoặc tại một vị trí `≥ j`; các vị trí bị bỏ qua đã được chứng minh thua. Khi `s[i+k] < s[j+k]`, với mọi `t ∈ [0, k]`, hậu tố `i + t` thua hậu tố `j + t` (cùng `t` ký tự đầu, rồi ký tự nhỏ hơn) ⇒ bỏ `i..i+k`. Đối xứng khi `>`. Vòng lặp dừng khi `j + k = n`: hậu tố `j` là tiền tố của hậu tố `i` nên không lớn hơn.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê cách chia | `O(C(n−1, k−1) · n)` | `O(n²)` |
| 2. Mọi điểm bắt đầu | `O(n · L)` | `O(L)` |
| 3. Hai con trỏ | `O(n)` | `O(1)` (+ kết quả) |
| 4. Băm + nhị phân LCP | `O(n log n)` | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'candidates': answer_candidates, 'two_pointers': answer_two_pointers, 'hash': answer_hash,
         'submit': lambda w, k: Solution().answerString(w, k)}
edge = [("a", 1, "a"), ("ab", 1, "ab"), ("ab", 2, "b"), ("abz", 2, "z"), ("zab", 2, "za"), ("aaa", 2, "aa"), ("bcbca", 2, "cbca")]
for w, k, exp in examples + edge:
    assert answer_brute(w, k) == exp, (w, k, answer_brute(w, k))
    for name, f in impls.items():
        assert f(w, k) == exp, (name, w, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
rng = random.Random(3406)
cnt = 0
for _ in range(2000):
    n = rng.randint(1, 9)
    w = ''.join(rng.choice('abc'[:rng.randint(1, 3)]) for _ in range(n))
    k = rng.randint(1, n)
    exp = answer_brute(w, k)
    for name, f in impls.items():
        assert f(w, k) == exp, (name, w, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

Khớp vét cạn trên 2000 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = 2·10^5`

In [13]:
import time
n = 2 * 10 ** 5
cases = {
    'ngẫu nhiên': (''.join(rng.choice('abcdefghijklmnopqrstuvwxyz') for _ in range(n)), 1000),
    'toàn z': ('z' * n, 2),
    'chu kỳ "zzy"': ('zzy' * (n // 3), 5),
    'gần giống nhau (xấu cho so sánh)': ('a' * (n - 1) + 'b', 3),
}
for name, (w, k) in cases.items():
    t = time.time(); r = Solution().answerString(w, k); t1 = time.time() - t
    t = time.time(); r2 = answer_hash(w, k); t2 = time.time() - t
    assert r == r2
    print(f"{name:34s}: độ dài kết quả {len(r):6d}, bắt đầu {r[:8]!r} | hai con trỏ {t1:.2f}s | băm {t2:.2f}s")

ngẫu nhiên                        : độ dài kết quả 131963, bắt đầu 'zzzwtrkl' | hai con trỏ 0.02s | băm 1.49s


toàn z                            : độ dài kết quả 199999, bắt đầu 'zzzzzzzz' | hai con trỏ 0.01s | băm 1.36s


chu kỳ "zzy"                      : độ dài kết quả 199994, bắt đầu 'zzyzzyzz' | hai con trỏ 0.01s | băm 1.24s


gần giống nhau (xấu cho so sánh)  : độ dài kết quả      1, bắt đầu 'b' | hai con trỏ 0.01s | băm 1.61s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên trường hợp `numFriends = 1`

In [14]:
def wrong_k1(word, k):
    i = last_substring_start(word)
    return word[i:i + len(word) - k + 1]

print('sai :', repr(wrong_k1("ab", 1)), '| đúng:', repr(Solution().answerString("ab", 1)))

sai : 'b' | đúng: 'ab'


**Vì sao sai:** với `k = 1` chỉ có một cách chia: chính `word`. Hậu tố lớn nhất `"b"` không phải là một mảnh.

### Lỗi 2 — Chỉ lấy ký tự lớn nhất

In [15]:
def wrong_max_char(word, k):
    return word if k == 1 else max(word)

print('sai :', repr(wrong_max_char("dbca", 2)), '| đúng:', repr(Solution().answerString("dbca", 2)))

sai : 'd' | đúng: 'dbc'


**Vì sao sai:** mảnh dài hơn bắt đầu bằng ký tự lớn nhất còn lớn hơn (tiền tố nhỏ hơn xâu dài).

### Lỗi 3 — Cắt sai độ dài (`n − k` thay vì `n − k + 1`)

In [16]:
def wrong_length(word, k):
    if k == 1:
        return word
    i = last_substring_start(word)
    return word[i:i + len(word) - k]

print('sai :', repr(wrong_length("dbca", 2)), '| đúng:', repr(Solution().answerString("dbca", 2)))

sai : 'db' | đúng: 'dbc'


**Vì sao sai:** `k − 1` mảnh còn lại cần ít nhất `k − 1` ký tự ⇒ mảnh lớn nhất dài tối đa `n − (k − 1) = n − k + 1`.

### Lỗi 4 — Chỉ xét các cửa sổ **đúng độ dài `L`**

In [17]:
def wrong_fixed_windows(word, k):
    if k == 1:
        return word
    L = len(word) - k + 1
    return max(word[i:i + L] for i in range(len(word) - L + 1))

print('sai :', repr(wrong_fixed_windows("abz", 2)), '| đúng:', repr(Solution().answerString("abz", 2)))

sai : 'bz' | đúng: 'z'


**Vì sao sai:** gần cuối xâu, mảnh có thể **ngắn hơn** `L` (ví dụ `"z"` ở cuối `"abz"`) và vẫn lớn hơn mọi cửa sổ dài `L`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xâu lớn nhất theo từ điển trong các xâu con có ràng buộc độ dài" ⇒ tiền tố của **hậu tố lớn nhất**.
- Hậu tố lớn nhất ⇒ hai con trỏ `i, j, k` (LC 1163) `O(n)`; hoặc thuật toán Duval / mảng hậu tố.
- Luôn kiểm tra trường hợp biên `k = 1`.
- So sánh xâu dài nhiều lần ⇒ cân nhắc băm + LCP.

## 15. Biến thể của bài

1. **LC 3403 — bản I:** `n ≤ 5000`, cách 2 là đủ.
2. **LC 1163 — Last Substring in Lexicographical Order:** chính là hậu tố lớn nhất.
3. Xâu **nhỏ nhất** theo từ điển trong hộp ⇒ cẩn thận: với `k ≥ 3` mọi ký tự đơn đều là mảnh (đáp án `min(word)`), nhưng với `k = 2` chỉ có tiền tố và hậu tố ⇒ `min(word[0], hậu tố thật nhỏ nhất)`.
4. Mảnh lớn nhất khi mỗi mảnh có độ dài **≥ m** ⇒ thay `k − 1` ký tự tối thiểu bằng `(k − 1)·m`.

In [18]:
# Biến thể 4: mỗi mảnh dài ít nhất m ký tự
def answer_min_len(word, k, m):
    n = len(word)
    if k * m > n:
        return None
    if k == 1:
        return word
    L = n - (k - 1) * m
    best = None
    for i in range(n):
        # phần trái i ký tự + phần phải phải chia được thành k-1 mảnh dài >= m (mỗi phía: 0 hoặc >= m ký tự/mảnh)
        for j in range(i + m, min(n, i + L) + 1):
            left, right = i, n - j
            ok = any(a * m <= left and b * m <= right and (a == 0) == (left == 0) and (b == 0) == (right == 0)
                     for a in range(k) for b in range(k) if a + b == k - 1)
            if ok and (best is None or word[i:j] > best):
                best = word[i:j]
    return best

def brute_min_len(word, k, m):
    n = len(word); best = None
    for cuts in combinations(range(1, n), k - 1):
        b = (0,) + cuts + (n,)
        parts = [word[x:y] for x, y in zip(b, b[1:])]
        if all(len(p) >= m for p in parts):
            mx = max(parts)
            best = mx if best is None or mx > best else best
    return best

rng = random.Random(15)
for _ in range(300):
    w = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 8))); k = rng.randint(1, 4); m = rng.randint(1, 3)
    assert answer_min_len(w, k, m) == brute_min_len(w, k, m), (w, k, m)
print('Biến thể "mỗi mảnh ≥ m" khớp vét cạn:', answer_min_len("dbcaab", 2, 2))

Biến thể "mỗi mảnh ≥ m" khớp vét cạn: dbca


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Liệt kê cách chia | mũ | Oracle |
| 2 | Mọi điểm bắt đầu | `O(n · L)` | Đủ cho bản I |
| 3 | Hậu tố lớn nhất hai con trỏ | `O(n)` | **Nộp bài** |
| 4 | Băm + nhị phân LCP | `O(n log n)` | Kiểm chứng độc lập |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `word = "aann"`, `k = 2`.

**Bài 2 (tính tay):** hậu tố lớn nhất của `"banana"`.

**Bài 3 (điền TODO):** viết `smallest_in_box(word, k)` — biến thể 3.

**Bài 4 (tự viết lại):** hậu tố lớn nhất bằng **mảng hậu tố** (nhân đôi + sắp xếp) cho `n` vừa.

**Bài 5 (mini challenge):** cho nhiều truy vấn `k` trên cùng `word`, trả lời mỗi truy vấn `O(1)` sau tiền xử lý.

In [19]:
def smallest_in_box_todo(word, k):
    # TODO
    raise NotImplementedError

def last_suffix_by_suffix_array_todo(s):
    # TODO
    raise NotImplementedError

def answer_many_k_todo(word, ks):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `L = 3`. (2) Hậu tố lớn nhất `"nn"`. (3) Cắt còn tối đa 3 ký tự ⇒ `"nn"`.

**Bài 2.** (1) Các hậu tố bắt đầu bằng `n`: `"nana"`, `"na"`. (2) `"nana" > "na"`. (3) `"nana"`.

**Bài 3.** (1) `k = 1` ⇒ `word`. (2) `k ≥ 3` ⇒ mọi ký tự đơn là mảnh ⇒ `min(word)`. (3) `k = 2` ⇒ mảnh là tiền tố hoặc hậu tố: so `word[0]` với hậu tố thật nhỏ nhất.

**Bài 4.** (1) Xếp hạng theo cặp `(rank[i], rank[i + 2^t])`. (2) Lặp tới khi hạng phân biệt. (3) Hậu tố có hạng lớn nhất.

**Bài 5.** (1) Hậu tố lớn nhất không phụ thuộc `k`. (2) Tính một lần. (3) Mỗi truy vấn cắt độ dài `n − k + 1` (riêng `k = 1`).

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().answerString("aann", 2) == answer_brute("aann", 2) == "nn"
assert "banana"[last_substring_start("banana"):] == "nana"
def smallest_in_box(word, k):
    if k == 1:
        return word
    if k >= 3:
        return min(word)
    return min([word[:1]] + [word[i:] for i in range(1, len(word))])
for w in ["dbca", "gggg", "zab", "bab", "cabca"]:
    for k in range(1, len(w) + 1):
        assert smallest_in_box(w, k) == min(all_pieces(w, k))
print('Bài 1: nn | Bài 2: nana | Bài 3 khớp vét cạn.')

Bài 1: nn | Bài 2: nana | Bài 3 khớp vét cạn.


In [21]:
# Bài 4, 5
def last_suffix_by_suffix_array(s):
    n = len(s)
    rank = [ord(c) for c in s]
    sa = list(range(n)); t = 1
    while True:
        key = lambda i: (rank[i], rank[i + t] if i + t < n else -1)
        sa.sort(key=key)
        new = [0] * n
        for idx in range(1, n):
            new[sa[idx]] = new[sa[idx - 1]] + (key(sa[idx]) != key(sa[idx - 1]))
        rank = new
        if rank[sa[-1]] == n - 1:
            break
        t *= 2
    return s[sa[-1]:]

def answer_many_k(word, ks):
    i = last_substring_start(word)
    n = len(word)
    return [word if k == 1 else word[i:i + n - k + 1] for k in ks]

rng = random.Random(19)
for _ in range(300):
    s = ''.join(rng.choice('abc') for _ in range(rng.randint(1, 30)))
    assert last_suffix_by_suffix_array(s) == max(s[i:] for i in range(len(s)))
w = ''.join(rng.choice('abc') for _ in range(8))
assert answer_many_k(w, list(range(1, 9))) == [answer_brute(w, k) for k in range(1, 9)]
print('Bài 4, 5 khớp; ví dụ', repr(w), '->', answer_many_k(w, [1, 2, 3, 8]))

Bài 4, 5 khớp; ví dụ 'cbbacbab' -> ['cbbacbab', 'cbbacba', 'cbbacb', 'c']


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
if k == 1: return word
i, j, t = 0, 1, 0
while j + t < n:
    if s[i+t] == s[j+t]: t += 1
    elif s[i+t] < s[j+t]: i += t + 1; t = 0; j = max(j, i + 1)
    else: j += t + 1; t = 0
return word[i : i + n - k + 1]
```

**Checklist:**

- [ ] `numFriends = 1` ⇒ trả về cả `word`?
- [ ] Độ dài cắt `n − k + 1`?
- [ ] Ứng viên gần cuối có thể ngắn hơn `L` (dùng hậu tố, không cửa sổ cố định)?
- [ ] Sau khi `i` nhảy, đảm bảo `j > i`?
- [ ] Đối chiếu chữ ký nếu có Premium?